# 04 · Bi-encoder: baseline B2 (zero-shot)

A bi-encoder encodes the query and the product **separately** into vectors; the score of a pair is the cosine similarity of the two vectors. Products are encoded once, independently of the query, which is what makes bi-encoders fast for retrieval. A cross-encoder (B3, Direction 1) reads query and product together and is usually more accurate for re-ranking; B2 measures that gap.

No fine-tuning: both models are used as published.

| Run | Model | Pooling | Product text |
|---|---|---|---|
| `b2_minilml6_title`, `b2_minilml6_main` | `sentence-transformers/all-MiniLM-L6-v2` | mean | `title`, `main` |
| `b2_bge_title`, `b2_bge_main` | `BAAI/bge-base-en-v1.5` (query instruction as in its model card) | CLS | `title`, `main` |

B2 is a baseline, so test is scored here as for B0, B1 and B3. Since the models never saw ESCI labels, their scores on **train** are not leaked either: `biencoder_features.parquet` holds them for every split and is an input of the LightGBM re-ranker (Direction 3).

**Kaggle settings:** GPU **T4**, **Internet on**.

**Inputs:**
- Dataset `wm-src` (`src/` and `data/results/`, see README). Required; `data/results/` holds `b1_bm25_main` for the paired comparison.
- Add Input → Notebook Output → **01_preprocess**. Required.

**How to run:**
1. `MODE = 'smoke'`: val pairs only (about 41k pairs), a few minutes. Check the verdict at the end of section 2.
2. `MODE = 'full'`: all 601k pairs (every product encoded once per model and text), about 1 hour; **Save Version → Save & Run All**.

**Output** (`/kaggle/working/`): `results/<run>.json`, `results/scores/<run>_{val,test}.parquet` (into `data/results/` in the repo), and `biencoder_features.parquet` (`example_id` + one column per run, all splits; stays on Kaggle).

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, os, time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer

from src.data import load_pairs, product_texts
from src.evaluation import RunLog

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': val pairs only | 'full': all pairs (train included, for the Direction 3 features)

MODELS = {
    'minilml6': {'model': 'sentence-transformers/all-MiniLM-L6-v2', 'pooling': 'mean', 'query_prefix': ''},
    'bge': {'model': 'BAAI/bge-base-en-v1.5', 'pooling': 'cls',
            'query_prefix': 'Represent this sentence for searching relevant passages: '},
}
FIELDS = ['title', 'main']
MAX_LEN = {'title': 128, 'main': 256}
QUERY_MAX_LEN = 64
BATCH_SIZE = 256
REF = 'b1_bm25_main'
OUT = Path('/kaggle/working')
RES = OUT / 'results'

assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE)

## 1. Data and encoding

Every query and every product is encoded once per model (and per product text); a pair's score is the dot product of the two normalised vectors, i.e. their cosine similarity. Texts are sorted by length before batching to reduce padding.

In [ ]:
pairs = load_pairs('title')
scope = pairs if MODE == 'full' else pairs[pairs['split'] == 'val']
scope = scope.reset_index(drop=True)
SPLITS = [s for s in ['val', 'test'] if (scope['split'] == s).any()]
MASK = {s: (scope['split'] == s).to_numpy() for s in SPLITS}

queries = scope.drop_duplicates('query_id')[['query_id', 'query']].reset_index(drop=True)
pids = pd.Index(scope['product_id'].unique())
TEXT = {f: product_texts(f).reindex(pids) for f in FIELDS}
assert all(t.notna().all() for t in TEXT.values())
q_row = pd.Index(queries['query_id']).get_indexer(scope['query_id'])
p_row = pids.get_indexer(scope['product_id'])
print(f'{len(scope):,} pairs | {len(queries):,} queries | {len(pids):,} products | splits {SPLITS}')


@torch.no_grad()
def encode(model, tok, texts, max_len, pooling, batch_size=BATCH_SIZE):
    # L2-normalised embeddings (float16 on CPU), in the order of `texts`
    texts = np.asarray(texts, dtype=object)
    order = np.argsort([len(t) for t in texts], kind='stable')
    out = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        enc = tok(list(texts[idx]), truncation=True, max_length=max_len, padding=True, return_tensors='pt').to('cuda')
        with torch.autocast('cuda', dtype=torch.float16):
            h = model(**enc).last_hidden_state
        if pooling == 'cls':
            e = h[:, 0]
        else:   # mean over real tokens
            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            e = (h * m).sum(1) / m.sum(1).clamp(min=1)
        out[idx] = F.normalize(e.float(), dim=-1).half().cpu().numpy()
    return out


def pair_scores(Q, P, chunk=100_000):
    # cosine similarity of every pair in `scope`
    s = np.empty(len(scope), dtype=np.float32)
    for i in range(0, len(scope), chunk):
        q = Q[q_row[i:i + chunk]].astype(np.float32)
        p = P[p_row[i:i + chunk]].astype(np.float32)
        s[i:i + chunk] = np.einsum('ij,ij->i', q, p)
    return s

## 2. Runs

Each run is logged with `RunLog` (`src/evaluation.py`) and compared with `b1_bm25_main` (loaded from `wm-src`) by paired bootstrap.

In [ ]:
log = RunLog(RES, {s: scope.loc[MASK[s], ['example_id', 'query_id', 'esci_label']] for s in SPLITS})
ref = REF if log.load(REF, splits=SPLITS) else None
print('paired comparison with', ref or 'nothing (b1_bm25_main not in the inputs)')

feats = scope[['example_id']].copy()
runs = []
for m, spec in MODELS.items():
    tok = AutoTokenizer.from_pretrained(spec['model'])
    model = AutoModel.from_pretrained(spec['model']).to('cuda').eval()
    t0 = time.time()
    Q = encode(model, tok, (spec['query_prefix'] + queries['query']).tolist(), QUERY_MAX_LEN, spec['pooling'])
    q_min = (time.time() - t0) / 60
    for f in FIELDS:
        t0 = time.time()
        P = encode(model, tok, TEXT[f].tolist(), MAX_LEN[f], spec['pooling'])
        enc_min = (time.time() - t0) / 60
        s = pair_scores(Q, P)
        name = f"{'smoke_' if MODE == 'smoke' else ''}b2_{m}_{f}"
        feats[name] = s
        log.log(name, {sp: s[MASK[sp]] for sp in SPLITS},
                {'mode': MODE, **spec, 'fields': f, 'max_len': MAX_LEN[f], 'query_max_len': QUERY_MAX_LEN,
                 'fine_tuned': False, 'n_products': len(pids), 'query_encode_minutes': round(q_min, 1),
                 'product_encode_minutes': round(enc_min, 1), 'gpu': torch.cuda.get_device_name(0)},
                compare_to=ref)
        runs.append(name)
        print(f'   encode: queries {q_min:.1f} min, products ({f}) {enc_min:.1f} min')
        del P
    del model, Q
    gc.collect()
    torch.cuda.empty_cache()

ok = all(log.records[n]['metrics']['val']['ndcg'] > 0.742 for n in runs)
print('Smoke test passed: switch MODE to full.' if MODE == 'smoke' and ok else
      'CHECK: a run is at random level (val nDCG <= 0.742)' if not ok else 'done')

## 3. Results

Reference on val: random order 0.742, BM25 title 0.816, BM25 main 0.823, MiniLM cross-encoder zero-shot 0.833, B3 fine-tuned 0.857.

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s', 'n_queries')))
log.print_paired()

## 4. Save the bi-encoder features for all splits

In [ ]:
if MODE == 'full':
    feats.to_parquet(OUT / 'biencoder_features.parquet', index=False)
for f in sorted(p for p in OUT.rglob('*') if p.is_file()):
    print(f'{str(f.relative_to(OUT)):46s} {f.stat().st_size / 1e6:7.2f} MB')